In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.isdir("smartstock-ai"):          # Colab/Kaggle: clone your repo first
    subprocess.run(["git", "clone", "https://github.com/<you>/smartstock-ai.git"], check=True); os.chdir("smartstock-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements.txt"], check=True)
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "backend")) and ROOT != "/": ROOT = os.path.dirname(ROOT)
os.chdir(ROOT); sys.path[:0] = [os.path.join(ROOT, "backend"), os.path.join(ROOT, "training", "scripts")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RAW_OK = os.path.exists("data/raw/m5/sales_train_validation.csv")
if not RAW_OK:   # no real M5 download present -> generate M5-format synthetic data so the notebook runs anywhere
    subprocess.run([sys.executable, "training/scripts/make_synthetic_m5.py"], check=True)
if not os.path.exists("data/processed/panel.npz"):
    subprocess.run([sys.executable, "training/scripts/prepare_data.py"], check=True)
def ensure_trained():
    if not os.path.exists("training/outputs/metrics.json"):
        subprocess.run([sys.executable, "training/scripts/train.py"], check=True)

# 01 · Data understanding
M5 files: `calendar.csv`, `sales_train_validation.csv` (wide: one column per day) and `sell_prices.csv`. If the real M5 files are not in `data/raw/m5/`, the first cell generates a synthetic dataset in the **same format**.

In [ ]:
cal = pd.read_csv("data/raw/m5/calendar.csv"); sales = pd.read_csv("data/raw/m5/sales_train_validation.csv"); prices = pd.read_csv("data/raw/m5/sell_prices.csv")
print("calendar", cal.shape, "| sales (wide)", sales.shape, "| prices", prices.shape)
display(sales.iloc[:3, :10]); display(cal.head(3)); display(prices.head(3))

In [ ]:
import json; print(json.dumps(json.load(open("data/processed/dq_report.json")), indent=2))   # data-quality report from prepare_data.py

**Observed vs simulated.** Demand, prices and calendar are *observed* (or synthetic stand-ins here). Lead times, costs, inventory and capacity are *simulated* later and always labelled as such.